In [1]:
import mysql.connector
import pandas as pd

conn = mysql.connector.connect(
    host = "localhost",
    user = "root",
    password = "sivakumar",
    database = "cart2insights"
)
cursor = conn.cursor()
print("connection successful")

connection successful


In [3]:
query = """
SELECT
    o.order_id,
    o.order_delivered_customer_date,
    o.order_estimated_delivery_date,
    r.review_score
FROM orders o
JOIN order_reviews r
    ON o.order_id = r.order_id
WHERE o.order_delivered_customer_date IS NOT NULL
  AND o.order_estimated_delivery_date IS NOT NULL
  AND r.review_score IS NOT NULL
"""

delivery_reviews = pd.read_sql(query, conn)

C:\Users\yendl\AppData\Local\Temp\ipykernel_19996\718470548.py:15: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  delivery_reviews = pd.read_sql(query, conn)


In [4]:
delivery_reviews["delivery_status"] = (
    delivery_reviews["order_delivered_customer_date"]
    > delivery_reviews["order_estimated_delivery_date"]
).map({
    True: "Delayed",
    False: "On Time"
})

In [5]:
display(
    delivery_reviews[
        ["delivery_status", "review_score"]
    ].head()
)

,delivery_status,review_score
0,Delayed,5
1,On Time,5
2,On Time,5
3,On Time,5
4,On Time,5


In [6]:
on_time_reviews = delivery_reviews[
    delivery_reviews["delivery_status"] == "On Time"
]["review_score"]

delayed_reviews = delivery_reviews[
    delivery_reviews["delivery_status"] == "Delayed"
]["review_score"]

In [7]:
print("On-time average review score:",
      on_time_reviews.mean())

print("Delayed average review score:",
      delayed_reviews.mean())

On-time average review score: 4.293577567732184
Delayed average review score: 2.566549798727438


In [9]:
from scipy.stats import ttest_ind

t_stat, p_value = ttest_ind(
    on_time_reviews,
    delayed_reviews,
    equal_var=False
)

print("T-statistic:", t_stat)
print("P-value:", p_value)

T-statistic: 89.55081413351809
P-value: 0.0


In [10]:
alpha = 0.05

if p_value < alpha:
    print("Reject the null hypothesis.")
    print("Delivery status has a statistically significant effect on review scores.")
else:
    print("Fail to reject the null hypothesis.")
    print("There is no statistically significant difference in review scores.")

Reject the null hypothesis.
Delivery status has a statistically significant effect on review scores.


In [13]:
query = """
SELECT
    p.product_category_name,
    oi.order_id,
    oi.price + oi.freight_value AS order_value
FROM order_items oi
JOIN products p
    ON oi.product_id = p.product_id
WHERE p.product_category_name IS NOT NULL
"""

category_orders = pd.read_sql(query, conn)

display(category_orders.head())

C:\Users\yendl\AppData\Local\Temp\ipykernel_19996\1478599655.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  category_orders = pd.read_sql(query, conn)


,product_category_name,order_id,order_value
0,perfumaria,f30149f4a8882a08895b6a242aa0d612,120.24
1,automotivo,f5eda0ded77c1293b04c953138c8331d,143.83
2,cama_mesa_banho,0bf736fd0fd5169d60de3699fcbcf986,242.10
3,utilidades_domesticas,3aba44d8e554ab4bb8c09f6f78032ca8,78.50
4,utilidades_domesticas,6f0dfb5b5398b271cc6bbd9ee263530e,78.50


In [14]:
category_counts = (
    category_orders["product_category_name"]
    .value_counts()
)

valid_categories = category_counts[
    category_counts >= 30
].index

anova_data = category_orders[
    category_orders["product_category_name"]
    .isin(valid_categories)
]

In [15]:
groups = [
    group["order_value"].values
    for name, group
    in anova_data.groupby("product_category_name")
]

In [16]:
from scipy.stats import f_oneway

f_stat, p_value = f_oneway(*groups)

print("F-statistic:", f_stat)
print("P-value:", p_value)

F-statistic: 206.94255846805365
P-value: 0.0


In [17]:
alpha = 0.05

if p_value < alpha:
    print("Reject the null hypothesis.")
    print("Average order value differs significantly across product categories.")
else:
    print("Fail to reject the null hypothesis.")
    print("No statistically significant difference was found.")

Reject the null hypothesis.
Average order value differs significantly across product categories.


In [19]:
query = """
SELECT
    op.payment_type,
    o.order_status
FROM order_payments op
JOIN orders o
    ON op.order_id = o.order_id
WHERE op.payment_type IS NOT NULL
  AND o.order_status IS NOT NULL
"""

payment_status = pd.read_sql(query, conn)

display(payment_status.head())

C:\Users\yendl\AppData\Local\Temp\ipykernel_19996\1052536474.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  payment_status = pd.read_sql(query, conn)


,payment_type,order_status
0,credit_card,delivered
1,credit_card,delivered
2,credit_card,delivered
3,credit_card,delivered
4,credit_card,delivered


In [20]:
contingency_table = pd.crosstab(
    payment_status["payment_type"],
    payment_status["order_status"]
)

display(contingency_table)

order_status,approved,canceled,created,delivered,invoiced,processing,shipped,unavailable
payment_type,,,,,,,,
boleto,0,95,2,19191,67,70,209,150
credit_card,2,444,3,74586,239,224,851,446
debit_card,0,7,0,1486,6,2,22,6
not_defined,0,3,0,0,0,0,0,0
voucher,0,115,0,5493,13,23,84,47


In [21]:
from scipy.stats import chi2_contingency

chi2_stat, p_value, degrees_freedom, expected = (
    chi2_contingency(contingency_table)
)

print("Chi-square statistic:", chi2_stat)
print("Degrees of freedom:", degrees_freedom)
print("P-value:", p_value)

Chi-square statistic: 677.0831369637666
Degrees of freedom: 28
P-value: 1.2047397488650834e-124


In [22]:
alpha = 0.05

if p_value < alpha:
    print("Reject the null hypothesis.")
    print("Payment method and order status are significantly associated.")
else:
    print("Fail to reject the null hypothesis.")
    print("No significant association was found.")

Reject the null hypothesis.
Payment method and order status are significantly associated.
